# Module 02: Scalar Quantization & Lloyd-Max

**Time**: ~35 minutes

**Prerequisites**: Module 01 (naive rounding destroys vector geometry)

**What you will learn**:
1. What scalar quantization is and why it matters
2. Why uniform quantization fails for bell-curve data
3. The Lloyd-Max algorithm -- the optimal scalar quantizer
4. How TurboQuant implements this in practice

---

In Module 01 you saw that rounding each coordinate to the nearest integer is a terrible quantizer.
The problem was not rounding per se -- it was that the rounding grid did not match the data distribution.
This module builds the right grid.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_close, check_intuition, visualize_1d_quantizer, section_header, run_quantize_experiment

import numpy as np
import torch
from scipy.stats import norm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. What Is Scalar Quantization?

Scalar quantization maps each continuous value to one of **K discrete levels**.

Think of reducing color depth in an image:
- A grayscale pixel can be any of 256 shades (8 bits)
- Reduce to 4 shades (2 bits) and you save 4x storage
- The trick is *which* 4 shades to keep

A scalar quantizer has two parts:
1. **Boundaries** (decision thresholds): where you split the number line
2. **Centroids** (reconstruction levels): what value you assign to each bin

For K levels you need K-1 boundaries and K centroids.

In [ ]:
section_header("Scalar Quantization", "Mapping continuous values to discrete levels")

# A simple example: 4 levels covering [-3, 3]
boundaries = np.array([-1.5, 0.0, 1.5])
centroids = np.array([-2.25, -0.75, 0.75, 2.25])

# Show what quantization does to a few values
test_values = np.array([-2.5, -0.3, 0.1, 1.8, 2.9])
# Assign each to a bin using searchsorted (same trick TurboQuant uses)
codes = np.searchsorted(boundaries, test_values)
reconstructed = centroids[codes]

print("Value -> Code -> Reconstructed")
for v, c, r in zip(test_values, codes, reconstructed):
    print(f"  {v:+.1f}  ->  {c}  ->  {r:+.2f}")

In [ ]:
# Visualize: a continuous signal quantized to 4 levels
np.random.seed(42)
x = np.linspace(-3, 3, 200)
signal = np.sin(x) + 0.3 * np.sin(3 * x)  # a smooth signal

codes = np.searchsorted(boundaries, signal)
quantized_signal = centroids[codes]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=signal, mode="lines", name="Original",
                         line=dict(color="steelblue", width=2)))
fig.add_trace(go.Scatter(x=x, y=quantized_signal, mode="lines", name="Quantized (4 levels)",
                         line=dict(color="red", width=2, dash="dot")))

# Show the 4 level lines
for c in centroids:
    fig.add_hline(y=c, line_dash="dash", line_color="lightgray", opacity=0.5)

fig.update_layout(title="Scalar Quantization: Continuous Signal -> 4 Discrete Levels",
                  xaxis_title="x", yaxis_title="Value",
                  template="plotly_white", height=400)
fig.show()

## 2. Uniform Quantization

The simplest approach: divide the range into **equal-width bins**.

This is what most people think of when they hear "quantization."

In [ ]:
section_header("Uniform Quantization", "Equal-width bins")

def uniform_quantizer(bits, lo=-3.0, hi=3.0):
    """Create a uniform scalar quantizer with 2^bits levels in [lo, hi]."""
    n_levels = 1 << bits
    # Centroids are evenly spaced
    centroids = torch.linspace(lo, hi, n_levels)
    # Boundaries are midpoints between adjacent centroids
    boundaries = (centroids[:-1] + centroids[1:]) / 2.0
    return boundaries, centroids

def quantize_scalar(x, boundaries, centroids):
    """Quantize tensor x using given boundaries and centroids."""
    codes = torch.searchsorted(boundaries, x)
    return codes, centroids[codes]

# 2-bit uniform quantizer
u_boundaries, u_centroids = uniform_quantizer(bits=2)
print(f"2-bit uniform quantizer:")
print(f"  Centroids:  {u_centroids.numpy()}")
print(f"  Boundaries: {u_boundaries.numpy()}")

In [ ]:
# Uniform quantization works well for... uniform data
torch.manual_seed(42)
uniform_data = torch.rand(10000) * 6 - 3  # Uniform[-3, 3]

codes_u, recon_u = quantize_scalar(uniform_data, u_boundaries, u_centroids)
mse_uniform_on_uniform = ((uniform_data - recon_u) ** 2).mean().item()
print(f"MSE of uniform quantizer on Uniform[-3,3] data: {mse_uniform_on_uniform:.4f}")

In [ ]:
# But real neural network weights/embeddings are NOT uniform -- they are bell-curved
torch.manual_seed(42)
normal_data = torch.randn(10000)  # N(0,1)

codes_n, recon_n = quantize_scalar(normal_data, u_boundaries, u_centroids)
mse_uniform_on_normal = ((normal_data - recon_n) ** 2).mean().item()
print(f"MSE of uniform quantizer on N(0,1) data: {mse_uniform_on_normal:.4f}")
print(f"\nThat is {mse_uniform_on_normal / mse_uniform_on_uniform:.1f}x worse!")

In [ ]:
# Why? Look at where the data actually lives vs where the bins are
fig = make_subplots(rows=1, cols=2, subplot_titles=["Uniform Data", "Normal Data"])

fig.add_trace(go.Histogram(x=uniform_data.numpy(), nbinsx=50, name="Uniform",
                           marker_color="steelblue", opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=normal_data.numpy(), nbinsx=50, name="Normal",
                           marker_color="coral", opacity=0.7), row=1, col=2)

# Show quantizer boundaries on both
for b in u_boundaries.numpy():
    fig.add_vline(x=b, line_dash="dash", line_color="red", row=1, col=1)
    fig.add_vline(x=b, line_dash="dash", line_color="red", row=1, col=2)

# Show centroids
for c in u_centroids.numpy():
    fig.add_vline(x=c, line_dash="dot", line_color="green", opacity=0.5, row=1, col=1)
    fig.add_vline(x=c, line_dash="dot", line_color="green", opacity=0.5, row=1, col=2)

fig.update_layout(title="Same Uniform Quantizer (red dashes = boundaries, green dots = centroids)",
                  template="plotly_white", height=400, showlegend=False)
fig.show()

print("\nThe problem: for normal data, most values cluster near 0.")
print("But the uniform quantizer gives equal space to the tails where almost nothing lives.")
print("Only 2 of 4 bins cover the range [-1.5, 1.5] where ~87% of the data is.")

## 3. The Key Insight: Match Your Bins to Your Data

If most values are near zero, **put more bins near zero**.

This is obvious once you see it, but the question is: *exactly* where should the boundaries and centroids go?

The answer is the **Lloyd-Max algorithm** -- it finds the boundaries and centroids that minimize the mean squared error (MSE) for a given distribution.

It works by alternating two steps:
1. **Given centroids, find optimal boundaries**: each boundary is the midpoint between adjacent centroids
2. **Given boundaries, find optimal centroids**: each centroid is the conditional mean of the data in its bin

This is essentially **k-means in 1D**, and it converges to a local (often global) optimum.

## 4. Build Lloyd-Max from Scratch

Let us implement Lloyd-Max for the standard normal distribution N(0,1).

### The math you need

For X ~ N(0,1), the conditional expectation in a bin [a, b] is:

$$E[X \mid a < X < b] = \frac{\phi(a) - \phi(b)}{\Phi(b) - \Phi(a)}$$

where:
- $\phi(x)$ = `norm.pdf(x)` is the standard normal PDF
- $\Phi(x)$ = `norm.cdf(x)` is the standard normal CDF

This formula comes from integrating $x \cdot \phi(x)$ over $[a, b]$ and dividing by the probability of the bin.

### The algorithm

```
Step 1: Start with some initial centroid guesses (e.g., linspace(-3, 3, K))
Step 2: Compute boundaries as midpoints between adjacent centroids
Step 3: Update each centroid to E[X | X in bin_i]
Step 4: Repeat until centroids stop moving
```

In [ ]:
section_header("Lloyd-Max Algorithm", "Building the optimal scalar quantizer from scratch")

def lloyd_max_normal(bits, max_iter=200, tol=1e-12, track_history=False):
    """Compute Lloyd-Max optimal quantizer for N(0,1).
    
    Args:
        bits: Number of quantization bits.
        max_iter: Maximum iterations.
        tol: Convergence tolerance.
        track_history: If True, return centroid history for visualization.
    
    Returns:
        boundaries, centroids, (history if track_history)
    """
    n_levels = 1 << bits  # 2^bits
    
    # Step 1: Initial guess -- spread centroids uniformly across [-3, 3]
    centroids = np.linspace(-3, 3, n_levels)
    history = [centroids.copy()] if track_history else None
    
    for iteration in range(max_iter):
        # Step 2: Boundaries = midpoints between adjacent centroids
        boundaries = (centroids[:-1] + centroids[1:]) / 2.0
        
        # Step 3: Update each centroid to E[X | X in bin_i]
        new_centroids = np.zeros(n_levels)
        # Add -inf and +inf as the outer edges
        edges = np.concatenate([[-np.inf], boundaries, [np.inf]])
        
        for i in range(n_levels):
            lo, hi = edges[i], edges[i + 1]
            
            # Probability that X falls in this bin
            prob = norm.cdf(hi) - norm.cdf(lo)
            
            if prob < 1e-15:
                # Bin is essentially empty -- keep centroid where it is
                new_centroids[i] = (
                    (lo + hi) / 2 
                    if np.isfinite(lo) and np.isfinite(hi) 
                    else centroids[i]
                )
            else:
                # E[X | lo < X < hi] = (phi(lo) - phi(hi)) / (Phi(hi) - Phi(lo))
                new_centroids[i] = (norm.pdf(lo) - norm.pdf(hi)) / prob
        
        if track_history:
            history.append(new_centroids.copy())
        
        # Step 4: Check convergence
        if np.max(np.abs(new_centroids - centroids)) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids
    
    # Final boundaries
    boundaries = (centroids[:-1] + centroids[1:]) / 2.0
    
    if track_history:
        return boundaries, centroids, history
    return boundaries, centroids

print("Lloyd-Max implementation ready.")

In [ ]:
# Run 2-bit Lloyd-Max and watch convergence
boundaries_lm, centroids_lm, history = lloyd_max_normal(bits=2, track_history=True)

print(f"Converged in {len(history) - 1} iterations")
print(f"Optimal centroids: {centroids_lm}")
print(f"Optimal boundaries: {boundaries_lm}")

In [ ]:
# Visualize convergence: watch centroids move to their optimal positions
fig = go.Figure()

n_levels = len(history[0])
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]

for level in range(n_levels):
    trajectory = [h[level] for h in history]
    iterations = list(range(len(trajectory)))
    fig.add_trace(go.Scatter(
        x=iterations, y=trajectory,
        mode="lines+markers",
        name=f"Centroid {level}",
        line=dict(color=colors[level % len(colors)], width=2),
        marker=dict(size=4),
    ))

fig.update_layout(
    title="Lloyd-Max Convergence: Centroids Moving to Optimal Positions",
    xaxis_title="Iteration",
    yaxis_title="Centroid Value",
    template="plotly_white",
    height=400,
)
fig.show()

print("\nNotice how quickly the centroids converge.")
print("The outer centroids move inward -- the tails do not need to be at +/-3.")
print("The inner centroids spread out to cover the high-density region near zero.")

## 5. Visualize: Uniform vs Lloyd-Max

Now let us see both quantizers on the same PDF using the `visualize_1d_quantizer` helper.

In [ ]:
section_header("Uniform vs Lloyd-Max", "Visual comparison on N(0,1)")

fig_uniform = visualize_1d_quantizer(
    boundaries=u_boundaries.numpy(),
    centroids=u_centroids.numpy(),
    title="Uniform Quantizer (2-bit) on N(0,1)",
)
fig_uniform.show()

In [ ]:
fig_lloydmax = visualize_1d_quantizer(
    boundaries=boundaries_lm,
    centroids=centroids_lm,
    title="Lloyd-Max Quantizer (2-bit) on N(0,1)",
)
fig_lloydmax.show()

print("Key differences:")
print(f"  Uniform centroids:   {u_centroids.numpy()}")
print(f"  Lloyd-Max centroids: {centroids_lm}")
print("\nLloyd-Max places centroids closer together near zero (high density)")
print("and does not waste levels on the far tails.")

## 6. Measure Distortion

Theory says Lloyd-Max should win. Let us measure it on 10,000 samples.

In [ ]:
section_header("Distortion Comparison", "Uniform vs Lloyd-Max on 10,000 N(0,1) samples")

torch.manual_seed(42)
samples = torch.randn(10000)

# Uniform quantizer
u_bounds, u_cents = uniform_quantizer(bits=2)
_, recon_uniform = quantize_scalar(samples, u_bounds, u_cents)
mse_uniform = ((samples - recon_uniform) ** 2).mean().item()

# Lloyd-Max quantizer
lm_bounds_t = torch.from_numpy(boundaries_lm).float()
lm_cents_t = torch.from_numpy(centroids_lm).float()
_, recon_lloydmax = quantize_scalar(samples, lm_bounds_t, lm_cents_t)
mse_lloydmax = ((samples - recon_lloydmax) ** 2).mean().item()

print(f"MSE (Uniform):   {mse_uniform:.6f}")
print(f"MSE (Lloyd-Max): {mse_lloydmax:.6f}")
print(f"\nLloyd-Max is {mse_uniform / mse_lloydmax:.1f}x better!")
print(f"Relative improvement: {(mse_uniform - mse_lloydmax) / mse_uniform * 100:.1f}%")

In [ ]:
# Compare across bit widths
print(f"{'Bits':<6} {'Uniform MSE':<14} {'Lloyd-Max MSE':<14} {'Improvement':<12}")
print("-" * 46)

for bits in [1, 2, 3, 4]:
    u_b, u_c = uniform_quantizer(bits)
    _, recon_u = quantize_scalar(samples, u_b, u_c)
    mse_u = ((samples - recon_u) ** 2).mean().item()
    
    lm_b, lm_c = lloyd_max_normal(bits)
    lm_b_t = torch.from_numpy(lm_b).float()
    lm_c_t = torch.from_numpy(lm_c).float()
    _, recon_lm = quantize_scalar(samples, lm_b_t, lm_c_t)
    mse_lm = ((samples - recon_lm) ** 2).mean().item()
    
    improvement = (mse_u - mse_lm) / mse_u * 100
    print(f"{bits:<6} {mse_u:<14.6f} {mse_lm:<14.6f} {improvement:<12.1f}%")

print("\nLloyd-Max consistently outperforms uniform quantization.")
print("The advantage is largest at low bit widths where every level counts.")

## 7. The Reference Implementation

You just built Lloyd-Max from scratch. Now let us look at how TurboQuant implements it.

The code lives in `reference-code/turboquant-torch/turboquant/codebook.py`.

In [ ]:
section_header("Reference Code", "turboquant/codebook.py")

# Let's read and display the reference implementation
print("""\
=== _lloyd_max_normal (the core algorithm) ===

def _lloyd_max_normal(bits, max_iter=200, tol=1e-12):
    n_levels = 1 << bits
    centroids = np.linspace(-3, 3, n_levels)

    for _ in range(max_iter):
        boundaries = (centroids[:-1] + centroids[1:]) / 2.0
        new_centroids = np.zeros(n_levels)
        edges = np.concatenate([[-np.inf], boundaries, [np.inf]])

        for i in range(n_levels):
            lo, hi = edges[i], edges[i + 1]
            prob = norm.cdf(hi) - norm.cdf(lo)
            if prob < 1e-15:
                new_centroids[i] = (
                    (lo + hi) / 2
                    if np.isfinite(lo) and np.isfinite(hi)
                    else centroids[i]
                )
            else:
                new_centroids[i] = (norm.pdf(lo) - norm.pdf(hi)) / prob

        if np.max(np.abs(new_centroids - centroids)) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids

    boundaries = (centroids[:-1] + centroids[1:]) / 2.0
    return Codebook(boundaries=boundaries, centroids=centroids)
""")

print("This is exactly what you implemented above. Let's walk through the key parts:")
print()
print("1. n_levels = 1 << bits")
print("   Bit shift: 1 << 2 = 4, 1 << 3 = 8. Fast way to compute 2^bits.")
print()
print("2. centroids = np.linspace(-3, 3, n_levels)")
print("   Initial guess: spread centroids uniformly across [-3, 3].")
print("   For N(0,1), 99.7% of values fall in this range.")
print()
print("3. boundaries = (centroids[:-1] + centroids[1:]) / 2.0")
print("   Nearest-neighbor rule: boundary between centroids is their midpoint.")
print()
print("4. edges = np.concatenate([[-np.inf], boundaries, [np.inf]])")
print("   The first bin goes from -inf to first boundary,")
print("   the last bin goes from last boundary to +inf.")
print()
print("5. (norm.pdf(lo) - norm.pdf(hi)) / prob")
print("   This IS the formula E[X | a < X < b] = (phi(a) - phi(b)) / (Phi(b) - Phi(a))")
print("   Applied to the standard normal. This is the heart of Lloyd-Max.")

In [ ]:
print("""\
=== LloydMaxCodebook (the runtime quantizer) ===

class LloydMaxCodebook:
    def __init__(self, bits, dim):
        self.bits = bits
        self.dim = dim
        cb = get_codebook(bits, dim)
        self._boundaries = torch.from_numpy(cb.boundaries).float()
        self._centroids = torch.from_numpy(cb.centroids).float()

    def quantize(self, x):
        boundaries = self._boundaries.to(x.device)
        return torch.searchsorted(boundaries, x)

    def dequantize(self, codes):
        centroids = self._centroids.to(codes.device)
        return centroids[codes]
""")

print("Key design points:")
print()
print("1. Codebook is precomputed (numpy, offline), then stored as torch tensors.")
print("   Lloyd-Max only runs once; quantize/dequantize are the hot path.")
print()
print("2. torch.searchsorted does the heavy lifting for quantization.")
print("   Given sorted boundaries, it finds which bin each value falls into.")
print("   This is O(n * log(K)) and fully vectorized on GPU.")
print()
print("3. Dequantization is just a table lookup: centroids[codes].")
print("   Each integer code maps to its centroid value.")
print()
print("4. get_codebook(bits, dim) handles scaling by 1/sqrt(d) for high dimensions")
print("   and switches to exact Beta codebooks for low dimensions.")
print("   We will explore this in a later module.")

In [ ]:
# Verify our implementation matches the reference
sys.path.insert(0, "../reference-code/turboquant-torch")
from turboquant.codebook import _lloyd_max_normal as ref_lloyd_max

for bits in [1, 2, 3, 4]:
    ref_cb = ref_lloyd_max(bits)
    our_b, our_c = lloyd_max_normal(bits)
    check_close(our_c, ref_cb.centroids, name=f"{bits}-bit centroids")
    check_close(our_b, ref_cb.boundaries, name=f"{bits}-bit boundaries")

print("\nOur implementation matches the reference exactly.")

## Exercise: 1-Bit Lloyd-Max

The simplest case: 1 bit = 2 levels. For N(0,1), the boundary is at 0 (by symmetry).

The two centroids should be:
- $E[X \mid X < 0] = -\frac{\phi(0)}{\Phi(0)} = -\frac{1/\sqrt{2\pi}}{1/2} \approx -0.7979$
- $E[X \mid X > 0] \approx +0.7979$

**Your task**: Run 1-bit Lloyd-Max and verify the centroids match $\pm 0.7979$.

In [ ]:
section_header("Exercise", "1-bit Lloyd-Max for N(0,1)")

# Run your lloyd_max_normal with bits=1
boundaries_1bit, centroids_1bit = lloyd_max_normal(bits=1)

print(f"1-bit boundary:  {boundaries_1bit}")
print(f"1-bit centroids: {centroids_1bit}")
print()

# The expected value: E[X | X > 0] = phi(0) / (1 - Phi(0)) = phi(0) / 0.5
expected_positive = norm.pdf(0) / 0.5  # = sqrt(2/pi) ~ 0.7979
expected_centroids = np.array([-expected_positive, expected_positive])
print(f"Expected centroids: {expected_centroids}")
print(f"sqrt(2/pi) = {np.sqrt(2 / np.pi):.6f}")
print()

# Check with helper
check_close(
    centroids_1bit, 
    expected_centroids, 
    name="1-bit Lloyd-Max centroids vs +/- sqrt(2/pi)"
)

In [ ]:
# Visualize the 1-bit quantizer
fig = visualize_1d_quantizer(
    boundaries=boundaries_1bit,
    centroids=centroids_1bit,
    title="1-Bit Lloyd-Max Quantizer: The Simplest Optimal Quantizer",
)
fig.show()

print("With just 1 bit, you can only distinguish positive vs negative.")
print(f"But the reconstruction values (+/- {expected_positive:.4f}) are optimal,")
print("not just +/- 1 or some arbitrary choice.")

## Summary

| Concept | Key Takeaway |
|---|---|
| Scalar quantization | Map each value to one of K levels |
| Uniform quantizer | Equal-width bins. Good for uniform data, bad for bell curves |
| Lloyd-Max | Iteratively optimizes bins for minimum MSE |
| Core formula | $E[X \mid a < X < b] = \frac{\phi(a) - \phi(b)}{\Phi(b) - \Phi(a)}$ for N(0,1) |
| In practice | Precompute codebook once, use `searchsorted` + table lookup at runtime |

**Next up**: Module 03 will show how to make the data *actually* look like N(0,1) using randomized Hadamard transforms, so that this optimal quantizer applies to real vectors.